# TL-Bot - char_classifier Training

**Before the very first run:**
- **Colab:** set runtime to GPU — *Runtime → Change runtime type → T4 GPU → Save*
- **Kaggle:** enable GPU (*Settings → Accelerator → GPU T4 x2*); add `RCLONE_TOKEN` secret (see below)
- **Lightning AI:** open a Studio with a T4 GPU before running this notebook
- **Local:** ensure `.venv` is active and a CUDA GPU is available (CPU works for smoke tests)

**Every session: run all cells top to bottom.**
- Cell 1 — set scripts and epoch count (platform is auto-detected).
- Cell 2 — mounts Drive (Colab), syncs checkpoints + dataset via rclone (Kaggle), confirms storage (Lightning/Local).
- Cell 3 — clones/pulls the repo and starts or resumes training. Child-process output is piped back into the cell; without that Colab sends it to the server log and failures surface as a bare `CalledProcessError`.
- Cell 4 — final results once cell 3 completes: run summary, the metrics of the epoch saved as `best.pt`, and `curves.png`. Refuses to report on an unfinished run.

Checkpoints are saved after every epoch and persist across sessions on all platforms.

---
**One-time: zip and upload the dataset**
```powershell
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset
# Colab/Kaggle: upload char-dataset.zip to My Drive/Colab Notebooks/TL-Bot/
# Lightning:    upload char-dataset.zip to /teamspace/studios/this_studio/TL-Bot/
# Local:        dataset is already present — no zip needed
```

**One-time: Kaggle rclone setup**
```powershell
# 1. Configure rclone (if not already done)
rclone config   # → New remote → name: gdrive → type: Google Drive → follow OAuth flow

# 2. Copy the token JSON to your clipboard
(Get-Content "$env:APPDATA\rclone\rclone.conf" | Select-String "^token = ").ToString().Replace("token = ", "") | Set-Clipboard

# 3. Add a Kaggle Secret: Add-ons → Secrets → Add new secret
#    Name: RCLONE_TOKEN   Value: paste clipboard (the {"access_token":...} JSON)
```

**Kaggle checkpoint sync:** `remote_train.py --sync-to` pushes checkpoints to Drive every 10 minutes during training and once on finish, so no manual sync cell is needed. To force a push after a crash:
`!rclone sync /kaggle/working/TL-Bot/checkpoints "gdrive:Colab Notebooks/TL-Bot/checkpoints/"`

---
## Cell 1 - Configure
Set the platform, scripts, and epoch count for this run. Edit here only.

In [ ]:
from pathlib import Path as _Path

def _detect_platform():
    import os
    # Use env var — KAGGLE_KERNEL_RUN_TYPE is set by Kaggle's infrastructure.
    # /kaggle/input alone is unreliable: the kaggle Python package creates it on other platforms too.
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if os.path.isdir("/teamspace/studios/this_studio"):
        return "lightning"
    return "local"

PLATFORM = _detect_platform()
print(f"Platform: {PLATFORM!r}")

# Scripts to train: "latin" | "kana" | "hangul" | "cjk" | "all"
# Single script -> checkpoints/<script>/   "all" -> checkpoints/
SCRIPTS = ["latin"]

# Epochs for this run.
# RESUME defaults to True for every script — checking progress.json before each
# session (cell 3 prints it automatically) is how to decide whether continuing
# is worthwhile or hyperparameters need a change, not a hardcoded per-script flag.
# best.pt only ever updates on a genuine score improvement (see train.py), so it
# can't regress from a bad resume or a bad fresh attempt either way.
EPOCHS = 60

SCHEDULER = "cosine"   # cosine (recommended) | cosine-warm | none
LR        = 3e-4       # head LR; backbone uses LR * 0.1
RESUME    = True        # resume last.pt; set to False only for a deliberate fresh restart

# --- Platform storage roots (only the one matching PLATFORM is used) ---

# Colab: Google Drive folder for checkpoints and dataset zip.
COLAB_ROOT = "/content/drive/MyDrive/Colab Notebooks/TL-Bot"

# Kaggle: rclone syncs checkpoints and dataset to/from the same Drive folder as Colab.
# See intro cell for one-time RCLONE_TOKEN setup.
KAGGLE_ROOT = "/kaggle/working/TL-Bot"

# Lightning AI: persistent studio storage path.
LIGHTNING_ROOT = "/teamspace/studios/this_studio/TL-Bot"

# Local: repo root (empty = cwd) and checkpoint output dir.
LOCAL_REPO = ""  # e.g. r"C:\Users\you\Documents\Discord-TL_Bot"
LOCAL_ROOT = str(_Path.home() / "tl-bot-checkpoints")

---
## Cell 2 - Setup
Mounts Drive (Colab), syncs checkpoints + dataset via rclone (Kaggle), or confirms local storage paths.

In [2]:
import os
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
elif PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient
    from pathlib import Path
    import subprocess
    import zipfile as _zipfile

    # Install rclone if not already present
    _which = subprocess.run(["which", "rclone"], capture_output=True)
    if _which.returncode != 0:
        print("Installing rclone ...")
        subprocess.run("curl -fsSL https://rclone.org/install.sh | sudo bash",
                       shell=True, check=True)
        print("rclone installed.")
    else:
        print(f"rclone already installed: {_which.stdout.decode().strip()}")

    # Configure gdrive via env vars — no config file needed.
    # RCLONE_TOKEN = the {"access_token":...} JSON from the "token = " line of rclone.conf.
    os.environ["RCLONE_CONFIG_GDRIVE_TYPE"]  = "drive"
    os.environ["RCLONE_CONFIG_GDRIVE_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_GDRIVE_TOKEN"] = UserSecretsClient().get_secret("RCLONE_TOKEN").strip()
    r = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
    if "gdrive:" not in r.stdout:
        raise RuntimeError("gdrive remote not found - check RCLONE_TOKEN secret.")
    print(f"rclone remotes: {r.stdout.strip()}")

    # Sync checkpoints from Drive (empty on first run is fine)
    ckpt_dst = Path(KAGGLE_ROOT) / "checkpoints"
    ckpt_dst.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(["rclone", "sync",
                             "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
                             str(ckpt_dst), "--progress"])
    if result.returncode != 0:
        print("Warning: checkpoint sync returned non-zero - continuing (may be first run).")
    else:
        print("Checkpoint sync complete.")

    # Pull dataset from Drive (skip if already extracted this session)
    ds_dir = Path(KAGGLE_ROOT) / "char-dataset"
    if not ds_dir.exists():
        ds_zip = Path(KAGGLE_ROOT) / "char-dataset.zip"
        print("Pulling char-dataset.zip from Drive ...")
        subprocess.run(["rclone", "copy",
                        "gdrive:Colab Notebooks/TL-Bot/char-dataset.zip",
                        str(Path(KAGGLE_ROOT)), "--progress"], check=True)
        with _zipfile.ZipFile(ds_zip, "r") as zf:
            zf.extractall(Path(KAGGLE_ROOT))
        ds_zip.unlink()
        print(f"Dataset ready: {ds_dir}")
    else:
        print(f"Dataset already present: {ds_dir}")
elif PLATFORM == "lightning":
    os.makedirs(LIGHTNING_ROOT, exist_ok=True)
    print(f"Storage ready: {LIGHTNING_ROOT}")
elif PLATFORM == "local":
    from pathlib import Path
    Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
    print(f"Repo  : {LOCAL_REPO or os.getcwd()}")
    print(f"Ckpts : {LOCAL_ROOT}")
else:
    raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}")

Mounted at /content/drive


---
## Cell 3 - Train
Clones or pulls the repo (Colab/Lightning), then starts or resumes training.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path as _Path

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"

if PLATFORM == "colab":
    REPO_DIR     = "/content/Discord-TL_Bot"
    STORAGE_ROOT = COLAB_ROOT
    storage_args = ["--storage-root", COLAB_ROOT]
elif PLATFORM == "kaggle":
    REPO_DIR     = "/kaggle/working/Discord-TL_Bot"
    STORAGE_ROOT = KAGGLE_ROOT
    storage_args = ["--storage-root", KAGGLE_ROOT, "--skip-dataset",
                    "--repo-dir", REPO_DIR,
                    "--sync-to", "gdrive:Colab Notebooks/TL-Bot/checkpoints/"]
elif PLATFORM == "lightning":
    REPO_DIR     = f"{LIGHTNING_ROOT}/Discord-TL_Bot"
    STORAGE_ROOT = LIGHTNING_ROOT
    storage_args = ["--storage-root", LIGHTNING_ROOT]
elif PLATFORM == "local":
    REPO_DIR     = LOCAL_REPO or os.getcwd()
    STORAGE_ROOT = LOCAL_ROOT
    storage_args = ["--storage-root", LOCAL_ROOT, "--skip-dataset",
                    "--repo-dir", REPO_DIR]
else:
    raise ValueError(f"Unknown PLATFORM: {PLATFORM!r}")

# Checkpoint dir, mirroring train.py's scoping. Resolved once here so cell 4 can
# reuse it instead of repeating the rule.
_ALL = {"latin", "kana", "hangul", "cjk"}
_sel = _ALL if "all" in SCRIPTS else set(SCRIPTS)
if _sel >= _ALL:
    CKPT_DIR = _Path(STORAGE_ROOT) / "checkpoints"
elif len(SCRIPTS) == 1:
    CKPT_DIR = _Path(STORAGE_ROOT) / "checkpoints" / SCRIPTS[0]
else:
    CKPT_DIR = _Path(STORAGE_ROOT) / "checkpoints" / "_".join(sorted(_sel))

# Clone / pull for remote platforms; local repo is already present.
if PLATFORM in ("colab", "lightning", "kaggle"):
    os.makedirs(REPO_DIR, exist_ok=True)
    if os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Kaggle: symlink dataset into repo tree (Models/Datasets/ is gitignored, won't exist after clone).
if PLATFORM == "kaggle":
    _ds_src = str(_Path(KAGGLE_ROOT) / "char-dataset")
    _ds_dst = f"{REPO_DIR}/Models/Datasets/char-dataset"
    if not os.path.exists(_ds_dst):
        os.makedirs(f"{REPO_DIR}/Models/Datasets", exist_ok=True)
        os.symlink(_ds_src, _ds_dst)
        print(f"Dataset linked: {_ds_src} -> {_ds_dst}")

# Print last training progress from progress.json before launching -- DO NOT REMOVE
# Wrapped: a display problem must never stop the training launch.
_prog = CKPT_DIR / "progress.json"
try:
    if _prog.exists():
        print("[progress]")
        for k, v in json.loads(_prog.read_text()).items():
            if not isinstance(v, (list, dict)):
                print(f"  {k}: {v}")
    else:
        print("[progress] No prior run found - starting fresh.")
except Exception as e:
    print(f"[progress] Could not read progress.json: {e}")


# Run a child process with its output streamed into the notebook.
#
# subprocess.run() without a pipe is useless here: IPython replaces sys.stdout at
# the Python level only, so a child inherits the kernel's real fd 1 and writes to
# Colab's server log, not this cell. Every setup message and traceback from
# remote_train.py was being discarded that way, leaving a bare CalledProcessError
# with no cause attached. Pipe it and re-print through sys.stdout instead.
def _run_streamed(cmd):
    print("$ " + " ".join(str(c) for c in cmd) + "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, errors="replace")
    tail = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        tail.append(line)
        del tail[:-40]
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f"remote_train.py exited {rc}. Last {len(tail)} lines:\n" + "".join(tail))
    return rc


_cmd = [
    "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
    "--skip-clone",
    "--scripts", *SCRIPTS,
    "--epochs", str(EPOCHS),
    "--scheduler", SCHEDULER,
    "--lr", str(LR),
    *storage_args,
]
if RESUME:
    _cmd.append("--resume")
_run_streamed(_cmd)

[progress]
  backbone: dinov2_vits14
  total_epochs: 60
  freeze_epochs: 3
  completed: 14
  epochs_remaining: 46
  phase: 2
  phase_label: backbone fine-tune
  best_score: 0.575835
  best_epoch: 9
  epochs_since_best: 5
  last_val_acc: 0.52108
  last_val_acc_delta: 0.023222
  last_f1: 0.570651
  last_precision: 0.674859
  last_recall: 0.521278
  last_lr: 0.00027779
  last_grad_norm: 8.7997
  last_epoch_secs: 923.8
  eta_secs: 42495
  saved_at: 2026-08-01T20:10:27
$ python -u /content/Discord-TL_Bot/Models/remote_train.py --skip-clone --scripts latin --epochs 60 --scheduler cosine --lr 0.0003 --storage-root /content/drive/MyDrive/Colab Notebooks/TL-Bot

 Remote Training Setup
  Scripts   : ['latin']
  Epochs    : 60  (freeze=3)
  Backbone  : dinov2_vits14
  Scheduler : cosine  mixup=0.2
  Ckpt dir  : /content/drive/MyDrive/Colab Notebooks/TL-Bot/checkpoints/latin
[setup] Drive already mounted.
[setup] Checking / installing packages ...

$ pip install -q wordninja lingua-language-detect

---
## Cell 4 - Final Results
Run once cell 3 finishes. Prints the run summary and the last epoch's metrics from `progress.json`, plus `curves.png`.

Says so and stops if the run has not reached its last epoch. Fields are read from the JSON as they come, so metrics added to `train.py` show up without editing this cell.

The test-set report (per-class precision/recall, top-1/3/5, confused pairs) is printed at the end of cell 3 and is not saved to disk.

In [ ]:
import json

# Final results, read from progress.json in CKPT_DIR (resolved in cell 3).
# Keys come from the file, so metrics added to train.py appear without edits here.

_d = json.loads((CKPT_DIR / "progress.json").read_text())
_hist = _d.get("history", [])

if _d.get("completed", 0) < _d.get("total_epochs", 0):
    print(f"[results] Training unfinished: epoch {_d.get('completed')} of "
          f"{_d.get('total_epochs')}. Re-run once cell 3 completes.")
else:
    print("=" * 72)
    print(f" FINAL RESULTS   {CKPT_DIR}")
    print("=" * 72)
    for k, v in _d.items():
        if not isinstance(v, (list, dict)):
            print(f"  {k:<20}: {v}")

    if _hist:
        print(f"\n  --- last epoch ({_hist[-1].get('epoch')}) ---")
        for k, v in _hist[-1].items():
            print(f"  {k:<20}: {v}")

    if (CKPT_DIR / "curves.png").exists():
        from IPython.display import Image, display
        display(Image(filename=str(CKPT_DIR / "curves.png")))